# Inference Production Runner

Run `inference_production` using the trained checkpoint and input parquet files in `data_v2`.

In [10]:
from pathlib import Path
import torch

# --- Paths (WSL) ---
repo_root = Path('.').resolve()
model_dir_wsl = Path('/mnt/c/Users/magno/Documents/GitHub/congestion-bogota/data/artifacts/model_gconvgru_2026_01_23')
data_v2_wsl = Path('/mnt/c/Users/magno/Documents/GitHub/congestion-bogota/data_v2')

if model_dir_wsl.exists() and data_v2_wsl.exists():
    model_dir = model_dir_wsl
    data_v2 = data_v2_wsl
    base_repo = Path('/mnt/c/Users/magno/Documents/GitHub/congestion-bogota')
else:
    model_dir = repo_root / 'data' / 'artifacts' / 'model_gconvgru_2026_01_23'
    data_v2 = repo_root / 'data_v2'
    base_repo = repo_root

model_path = model_dir / 'model.pth'

# Data inputs (parquets)
jams_path = data_v2 / 'df_jams_test.parquet'
irregularities_path = data_v2 / 'df_irregularidades_test.parquet'
weather_path = data_v2 / 'df_clima_test.parquet'
pmt_puntos_path = data_v2 / 'df_pmt_puntos_test.parquet'
pmt_tramos_path = data_v2 / 'df_pmt_tramos_test.parquet'
events_path = data_v2 / 'df_eventos_test.parquet'

# Shared assets (graph/static/norm) from training preprocess
processed_data_dir = base_repo / 'data' / 'datos_procesados'
graph_path = processed_data_dir / 'conectivity_graph' / 'road_graph.pickle'
static_path = processed_data_dir / 'sparse_datasets' / 'static_dataset.h5'
norm_path = processed_data_dir / 'sparse_datasets' / 'normalization_constants.pkl'

# Optional output path
output_path = base_repo / 'data' / 'artifacts' / 'inference_preds.npy'

for p in [model_path, jams_path, irregularities_path, weather_path, pmt_puntos_path, pmt_tramos_path, events_path, graph_path, static_path, norm_path]:
    if not p.exists():
        raise FileNotFoundError(f'Missing required file: {p}')

print('Paths OK')


Paths OK


In [11]:
keys_to_drop = {"include_weather", "num_weather_features", "include_velocity_target"}
for k in keys_to_drop:
    model_kwargs.pop(k, None)

AttributeError: 'bool' object has no attribute 'pop'

AttributeError: 'bool' object has no attribute 'pop'

In [6]:
model_kwargs

{'static_dim': 117,
 'dynamic_dim': 126,
 'future_dim': 5,
 'input_seq_len': 150,
 'forecast_horizon': 50,
 'model_type': 'GConvGRU',
 'include_velocity_target': True,
 'include_weather': True,
 'num_weather_features': 4}

In [5]:
import sys
import pickle
import torch
from pathlib import Path

cwd = Path.cwd().resolve()
if cwd.name == 'notebooks':
    repo_root = cwd.parent
else:
    repo_root = cwd

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.models.gnn_temporal import GNN_Temporal_Model

# --- Load checkpoint ---
checkpoint = torch.load(model_path, map_location='cpu')
model_kwargs = checkpoint.get('model_kwargs', {})
state_dict = checkpoint.get('state_dict', None)

if state_dict is None:
    raise ValueError('Checkpoint missing state_dict. Cannot load model weights.')

# Build model
model = GNN_Temporal_Model(
    edge_index=None,
    **model_kwargs,
)
model.load_state_dict(state_dict)

# Move to device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
model.eval()

print('Model loaded:', model.__class__.__name__, 'device=', device)
print('model_kwargs:', model_kwargs)


/tmp/ipykernel_5479/1433788254.py:18: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(model_path, map_location='cpu')


TypeError: GNN_Temporal_Model.__init__() got an unexpected keyword argument 'include_weather'

In [3]:
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
if cwd.name == 'notebooks':
    repo_root = cwd.parent
else:
    repo_root = cwd

src_path = repo_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from src.inference_production import InferencePipeline

# Initialize pipeline without auto-loading model
pipeline = InferencePipeline(
    graph_path=str(graph_path),
    static_data_path=str(static_path),
    normalization_constants_path=str(norm_path),
    model_path=None,
    use_ray=False,
)

# Inject the model we loaded from checkpoint
pipeline.model = model

results = pipeline.predict(
    jams_path=str(jams_path),
    irregularities_path=str(irregularities_path),
    weather_path=str(weather_path),
    pmt_puntos_path=str(pmt_puntos_path),
    pmt_tramos_path=str(pmt_tramos_path),
    events_path=str(events_path),
    output_path=str(output_path),
)

print('Results keys:', results.keys())
print('num_batches:', results['num_batches'])
print('num_nodes:', results['num_nodes'])
print('load_time:', results['load_time'])
print('inference_time:', results['inference_time'])
print('total_time:', results['total_time'])

if results['predictions'] is not None:
    pred = results['predictions'][0]
    print('First prediction shape:', pred.shape)


2026-01-22 19:15:55,037 - src.inference_production - INFO - Initializing inference pipeline...
2026-01-22 19:16:06,064 - src.inference_production - INFO - Pipeline initialized with 139107 nodes
2026-01-22 19:16:06,065 - src.inference_production - INFO - Starting inference...
2026-01-22 19:16:06,066 - src.inference_production - INFO - Loading data...
2026-01-22 19:16:58,163 - src.inference_production - INFO - Data loaded in 52.10s - 1 batch(es)
2026-01-22 19:16:58,233 - src.inference_production - INFO - Running model inference...
2026-01-22 19:16:58,237 - src.inference_production - ERROR - Inference failed: GNN_Temporal_Model.forward() got an unexpected keyword argument 'x_static'
Traceback (most recent call last):
  File "/mnt/c/Users/magno/Documents/GitHub/congestion-bogota/src/inference_production.py", line 120, in predict
    pred = self.model(
  File "/home/jose_marquez/miniconda3/envs/traffic-gnn-v2/lib/python3.10/site-packages/torch/nn/modules/module.py", line 1736, in _wrapped_c

TypeError: GNN_Temporal_Model.forward() got an unexpected keyword argument 'x_static'